# Classificação de Falhas em Motores Turbofan — CMAPSS FD001

**Disciplina:** Machine Learning — Especialização em Inteligência Artificial

**Dataset:** NASA CMAPSS FD001 (Turbofan Engine Degradation Simulation)

**Objetivo:** Classificar falha iminente em motores turbofan com base em séries temporais de sensores. A variável-alvo é binária: `1` se a vida útil restante (RUL) ≤ 30 ciclos (`falha_iminente`), `0` caso contrário.

**Abordagem:** Classificação supervisionada com **XGBoost** utilizando um subconjunto reduzido de **6 sensores** selecionados por correlação com o target:
- `sensor_04` → temp_turbina_c
- `sensor_07` → pressao_compressor_psi
- `sensor_11` → pressao_estatica_psi
- `sensor_12` → vazao_combustivel
- `sensor_15` → razao_bypass
- `sensor_21` → temp_mancal_c

O notebook cobre desde o carregamento e análise exploratória até o treino, avaliação e exportação do modelo.
\n**Modelos comparados:** XGBoost (principal) vs Random Forest (baseline)\n

In [1]:
# ============================================================
# 2. Importação de bibliotecas
# ============================================================
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import json as json_lib
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix
)
import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier
import joblib

warnings.filterwarnings("ignore")
np.random.seed(42)

print("Todas as bibliotecas importadas com sucesso.")

Todas as bibliotecas importadas com sucesso.


In [2]:
# ============================================================
# 3. Carregamento dos dados
# ============================================================
# Nomes das 26 colunas conforme documentação do CMAPSS
colunas = (
    ["id_motor", "ciclo", "op_setting_1", "op_setting_2", "op_setting_3"]
    + [f"sensor_{i:02d}" for i in range(1, 22)]
)
print(f"{len(colunas)} colunas definidas")

BASE = "../data"
train = pd.read_csv(f"{BASE}/train_FD001.txt", sep="\\s+", header=None, names=colunas)

print(f"Shape: {train.shape}")
print(f"Motores únicos: {train.id_motor.nunique()}")
print(f"Nulos totais: {train.isnull().sum().sum()}")
print(f"Duplicatas: {train.duplicated().sum()}")
print("\n--- info() ---")
train.info()
print("\n--- describe() ---")
train.describe()

26 colunas definidas


Shape: (20631, 26)
Motores únicos: 100
Nulos totais: 0
Duplicatas: 0

--- info() ---
<class 'pandas.DataFrame'>
RangeIndex: 20631 entries, 0 to 20630
Data columns (total 26 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id_motor      20631 non-null  int64  
 1   ciclo         20631 non-null  int64  
 2   op_setting_1  20631 non-null  float64
 3   op_setting_2  20631 non-null  float64
 4   op_setting_3  20631 non-null  float64
 5   sensor_01     20631 non-null  float64
 6   sensor_02     20631 non-null  float64
 7   sensor_03     20631 non-null  float64
 8   sensor_04     20631 non-null  float64
 9   sensor_05     20631 non-null  float64
 10  sensor_06     20631 non-null  float64
 11  sensor_07     20631 non-null  float64
 12  sensor_08     20631 non-null  float64
 13  sensor_09     20631 non-null  float64
 14  sensor_10     20631 non-null  float64
 15  sensor_11     20631 non-null  float64
 16  sensor_12     20631 non-null  float64


,id_motor,ciclo,op_setting_1,op_setting_2,op_setting_3,sensor_01,sensor_02,sensor_03,sensor_04,sensor_05,...,sensor_12,sensor_13,sensor_14,sensor_15,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21
count,20631.000000,20631.000000,20631.000000,20631.000000,20631.0,20631.00,20631.000000,20631.000000,20631.000000,2.063100e+04,...,20631.000000,20631.000000,20631.000000,20631.000000,2.063100e+04,20631.000000,20631.0,20631.0,20631.000000,20631.000000
mean,51.506568,108.807862,-0.000009,0.000002,100.0,518.67,642.680934,1590.523119,1408.933782,1.462000e+01,...,521.413470,2388.096152,8143.752722,8.442146,3.000000e-02,393.210654,2388.0,100.0,38.816271,23.289705
std,29.227633,68.880990,0.002187,0.000293,0.0,0.00,0.500053,6.131150,9.000605,5.329200e-15,...,0.737553,0.071919,19.076176,0.037505,3.469531e-18,1.548763,0.0,0.0,0.180746,0.108251
min,1.000000,1.000000,-0.008700,-0.000600,100.0,518.67,641.210000,1571.040000,1382.250000,1.462000e+01,...,518.690000,2387.880000,8099.940000,8.324900,3.000000e-02,388.000000,2388.0,100.0,38.140000,22.894200
25%,26.000000,52.000000,-0.001500,-0.000200,100.0,518.67,642.325000,1586.260000,1402.360000,1.462000e+01,...,520.960000,2388.040000,8133.245000,8.414900,3.000000e-02,392.000000,2388.0,100.0,38.700000,23.221800
50%,52.000000,104.000000,0.000000,0.000000,100.0,518.67,642.640000,1590.100000,1408.040000,1.462000e+01,...,521.480000,2388.090000,8140.540000,8.438900,3.000000e-02,393.000000,2388.0,100.0,38.830000,23.297900
75%,77.000000,156.000000,0.001500,0.000300,100.0,518.67,643.000000,1594.380000,1414.555000,1.462000e+01,...,521.950000,2388.140000,8148.310000,8.465600,3.000000e-02,394.000000,2388.0,100.0,38.950000,23.366800
max,100.000000,362.000000,0.008700,0.000600,100.0,518.67,644.530000,1616.910000,1441.490000,1.462000e+01,...,523.380000,2388.560000,8293.720000,8.584800,3.000000e-02,400.000000,2388.0,100.0,39.430000,23.618400


In [3]:
# ============================================================
# 4. Engenharia de features e seleção de sensores
# ============================================================
# Calcular RUL para cada motor
train["rul"] = train.groupby("id_motor")["ciclo"].transform("max") - train["ciclo"]

# Target binário: 1 se RUL <= 30 (falha iminente)
train["falha_iminente"] = (train["rul"] <= 30).astype(int)

print("Distribuição do target:")
print(train["falha_iminente"].value_counts())
print(f"\nProporção falha: {train['falha_iminente'].mean():.3f}")

# Correlação dos sensores com falha_iminente
sensor_cols = [c for c in train.columns if "sensor_" in c]
corr_target = (
    train[sensor_cols + ["falha_iminente"]]
    .corr()["falha_iminente"]
    .drop("falha_iminente")
    .abs()
    .sort_values(ascending=False)
)

print("\nCorrelação absoluta dos sensores com falha_iminente:")
print(corr_target.to_string())

# Selecionar os 6 sensores mais correlacionados
sensores_selecionados = [
    "sensor_04", "sensor_07", "sensor_11",
    "sensor_12", "sensor_15", "sensor_21"
]

# Nomes amigáveis
nomes_amigaveis = {
    "sensor_04": "temp_turbina_c",
    "sensor_07": "pressao_compressor_psi",
    "sensor_11": "pressao_estatica_psi",
    "sensor_12": "vazao_combustivel",
    "sensor_15": "razao_bypass",
    "sensor_21": "temp_mancal_c"
}

print(f"\n{len(sensores_selecionados)} sensores selecionados:")
for s in sensores_selecionados:
    corr_val = train[s].corr(train["falha_iminente"])
    print(f"  {s:12s} → {nomes_amigaveis[s]:30s}  (corr={corr_val:.4f})")

Distribuição do target:
falha_iminente
0    17531
1     3100
Name: count, dtype: int64

Proporção falha: 0.150

Correlação absoluta dos sensores com falha_iminente:
sensor_11    0.665655
sensor_04    0.648406
sensor_12    0.640174
sensor_07    0.625592
sensor_15    0.618732
sensor_21    0.606480
sensor_20    0.599912
sensor_17    0.583067
sensor_02    0.581404
sensor_03    0.561892
sensor_08    0.542910
sensor_13    0.539915
sensor_09    0.419539
sensor_14    0.341063
sensor_06    0.059579
sensor_01         NaN
sensor_05         NaN
sensor_10         NaN
sensor_16         NaN
sensor_18         NaN
sensor_19         NaN

6 sensores selecionados:
  sensor_04    → temp_turbina_c                  (corr=0.6484)
  sensor_07    → pressao_compressor_psi          (corr=-0.6256)
  sensor_11    → pressao_estatica_psi            (corr=0.6657)
  sensor_12    → vazao_combustivel               (corr=-0.6402)
  sensor_15    → razao_bypass                    (corr=0.6187)
  sensor_21    → temp_mancal_c

In [4]:
# ============================================================
# 5. Análise Exploratória (EDA) — 6 gráficos
# ============================================================
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 150

OUT = "../report"
os.makedirs(OUT, exist_ok=True)

# Dicionário de cores
cores = ["#2ecc71", "#e74c3c"]

# --- Gráfico 1: Distribuição do target ---
fig, ax = plt.subplots(figsize=(8, 5))
train["falha_iminente"].value_counts().sort_index().plot(
    kind="bar", color=cores, ax=ax, edgecolor="black"
)
ax.set_title("Distribuição das Classes", fontsize=14, fontweight="bold")
ax.set_xticks([0, 1])
ax.set_xticklabels(["Normal (RUL > 30)", "Falha Iminente (RUL ≤ 30)"])
ax.set_ylabel("Quantidade de Amostras")
for i, v in enumerate(train["falha_iminente"].value_counts().sort_index()):
    ax.text(i, v + 50, str(v), ha="center", fontweight="bold")
plt.tight_layout()
plt.savefig(f"{OUT}/01_distribuicao_target.png", dpi=150)
plt.close()

# --- Gráfico 2: RUL ao longo dos ciclos — Motor #5 ---
m5 = train[train.id_motor == 5]
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(m5["ciclo"], m5["rul"], "b-", linewidth=2, label="RUL")
ax.axhline(y=30, color="r", ls="--", alpha=0.8, linewidth=2,
           label="Limiar de Falha (30 ciclos)")
ax.fill_between(m5["ciclo"], 0, 30, alpha=0.12, color="r", label="Zona de Falha Iminente")
ax.set_xlabel("Ciclo de Operação")
ax.set_ylabel("Vida Útil Restante (RUL)")
ax.set_title("Evolução da Degradação — Motor #5", fontsize=14, fontweight="bold")
ax.legend(loc="upper right")
plt.tight_layout()
plt.savefig(f"{OUT}/02_degradacao_motor.png", dpi=150)
plt.close()

# --- Gráfico 3: Matriz de Correlação (sensores selecionados + target) ---
cols_corr = sensores_selecionados + ["falha_iminente"]
plt.figure(figsize=(9, 7))
corr_mat = train[cols_corr].corr()
mask = np.triu(np.ones_like(corr_mat, dtype=bool), k=1)
sns.heatmap(corr_mat, mask=mask, cmap="RdBu_r", center=0,
            square=True, linewidths=0.8, annot=True, fmt=".3f",
            cbar_kws={"shrink": 0.8})
plt.title("Matriz de Correlação — 6 Sensores + Target", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{OUT}/03_matriz_correlacao.png", dpi=150)
plt.close()

# --- Gráfico 4: Boxplots dos 6 sensores vs falha ---
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for i, sensor in enumerate(sensores_selecionados):
    ax = axes[i // 3, i % 3]
    sns.boxplot(data=train, x="falha_iminente", y=sensor, ax=ax,
                palette=cores, width=0.6)
    corr_val = train[sensor].corr(train["falha_iminente"])
    ax.set_title(f"{nomes_amigaveis[sensor]} (corr={corr_val:.3f})", fontweight="bold")
    ax.set_xlabel("")
    ax.set_xticklabels(["Normal", "Falha"])
fig.suptitle("Boxplots dos 6 Sensores vs Falha Iminente",
             fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(f"{OUT}/04_boxplots_sensores.png", dpi=150)
plt.close()

# --- Gráfico 5: Tendência do sensor mais correlacionado para 4 motores ---
top_sensor = corr_target.index[0]
top_nome = nomes_amigaveis.get(top_sensor, top_sensor)
fig, ax = plt.subplots(figsize=(12, 5))
motores_exemplo = [1, 5, 10, 15]
palette = plt.cm.Set1
for idx, mid in enumerate(motores_exemplo):
    d = train[train.id_motor == mid]
    ax.plot(d["ciclo"], d[top_sensor], color=palette(idx / len(motores_exemplo)),
            linewidth=2, label=f"Motor {mid}")
ax.set_xlabel("Ciclo de Operação")
ax.set_ylabel(top_nome)
ax.set_title(f"Tendência do Sensor Mais Correlacionado — {top_nome} ({top_sensor})",
             fontsize=13, fontweight="bold")
ax.legend()
plt.tight_layout()
plt.savefig(f"{OUT}/05_tendencia_top_sensor.png", dpi=150)
plt.close()

# --- Gráfico 6: Histograma de ciclos até falha por motor ---
max_ciclos = train.groupby("id_motor")["ciclo"].max()
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(max_ciclos, bins=20, edgecolor="black", color="#3498db", alpha=0.75)
mediana = max_ciclos.median()
ax.axvline(mediana, color="r", ls="--", linewidth=2,
           label=f"Mediana: {mediana:.0f} ciclos")
ax.set_title("Distribuição de Ciclos até Falha por Motor", fontsize=14, fontweight="bold")
ax.set_xlabel("Ciclos Totais de Operação")
ax.set_ylabel("Número de Motores")
ax.legend()
plt.tight_layout()
plt.savefig(f"{OUT}/06_ciclos_ate_falha.png", dpi=150)
plt.close()

print("Todos os 6 gráficos salvos em ../report/")

Todos os 6 gráficos salvos em ../report/


In [5]:
# ============================================================
# 6. Pré-processamento
# ============================================================
# Features: apenas os 6 sensores selecionados
feature_cols = sensores_selecionados.copy()
print(f"Features: {feature_cols}")
print(f"Nomes amigáveis: {[nomes_amigaveis[c] for c in feature_cols]}")

X = train[feature_cols].copy()
y = train["falha_iminente"].copy()

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s = scaler.transform(X_val)

print(f"\nConjuntos:")
print(f"  X_train: {X_train_s.shape}")
print(f"  X_val:   {X_val_s.shape}")
print(f"  y_train: {y_train.value_counts().to_dict()}")
print(f"  y_val:   {y_val.value_counts().to_dict()}")

Features: ['sensor_04', 'sensor_07', 'sensor_11', 'sensor_12', 'sensor_15', 'sensor_21']
Nomes amigáveis: ['temp_turbina_c', 'pressao_compressor_psi', 'pressao_estatica_psi', 'vazao_combustivel', 'razao_bypass', 'temp_mancal_c']

Conjuntos:
  X_train: (16504, 6)
  X_val:   (4127, 6)
  y_train: {0: 14024, 1: 2480}
  y_val:   {0: 3507, 1: 620}


In [6]:
# ============================================================
# 7. Modelagem — 2 modelos
# ============================================================

# ------ 7.1 XGBoost (modelo principal) ------
print('\n' + '=' * 60)
print('Treinando XGBoost...')
print('=' * 60)
pos_scale = (y_train == 0).sum() / (y_train == 1).sum()
xgb_model = xgb.XGBClassifier(
    n_estimators=150, max_depth=6, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=pos_scale, random_state=42, eval_metric='logloss'
)
xgb_model.fit(X_train_s, y_train)
y_pred_xgb = xgb_model.predict(X_val_s)
y_prob_xgb = xgb_model.predict_proba(X_val_s)[:, 1]
print('XGBoost OK')

# ------ 7.2 Random Forest ------
print('\n' + '=' * 60)
print('Treinando Random Forest...')
print('=' * 60)
rf_model = RandomForestClassifier(
    n_estimators=100, max_depth=15, random_state=42, n_jobs=-1
)
rf_model.fit(X_train_s, y_train)
y_pred_rf = rf_model.predict(X_val_s)
y_prob_rf = rf_model.predict_proba(X_val_s)[:, 1]
print('Random Forest OK')



Treinando XGBoost...


XGBoost OK

Treinando Random Forest...


Random Forest OK


In [7]:
# ============================================================
# 8. Avaliacao comparativa — 2 modelos
# ============================================================

def calc_metrics(nome, yt, yp, ypr):
    return {'Modelo': nome,
            'Acuracia': round(accuracy_score(yt, yp), 4),
            'Precisao': round(precision_score(yt, yp), 4),
            'Recall': round(recall_score(yt, yp), 4),
            'F1-Score': round(f1_score(yt, yp), 4),
            'AUC-ROC': round(roc_auc_score(yt, ypr), 4)}

df_res = pd.DataFrame([
    calc_metrics('Random Forest', y_val, y_pred_rf, y_prob_rf),
    calc_metrics('XGBoost', y_val, y_pred_xgb, y_prob_xgb),
])

print('=' * 70)
print('           TABELA COMPARATIVA')
print('=' * 70)
print(df_res.to_string(index=False))

# Matrizes de confusao
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (nome, yp) in zip(axes, [('Random Forest', y_pred_rf), ('XGBoost', y_pred_xgb)]):
    sns.heatmap(confusion_matrix(y_val, yp), annot=True, fmt="d", cmap="Blues", ax=ax, cbar=False)
    ax.set_title(nome)
fig.suptitle('Matrizes de Confusao', fontsize=14)
plt.tight_layout()
plt.savefig(f'{OUT}/07_matrizes_confusao.png', dpi=150)
plt.close()

# Curva ROC comparativa
plt.figure(figsize=(10, 7))
for nome, ypr in [('Random Forest', y_prob_rf), ('XGBoost', y_prob_xgb)]:
    fpr, tpr, _ = roc_curve(y_val, ypr)
    plt.plot(fpr, tpr, label=f'{nome} (AUC={roc_auc_score(y_val, ypr):.3f})')
plt.plot([0,1],[0,1],'k--',alpha=0.5,label='Aleatorio')
plt.xlabel('FPR'); plt.ylabel('TPR')
plt.title('Curva ROC'); plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUT}/08_curva_roc.png', dpi=150)
plt.close()

# Feature importance XGBoost
plt.figure(figsize=(10, 8))
imp = pd.DataFrame({"f": feature_cols, "i": xgb_model.feature_importances_}).sort_values("i")[::-1]
plt.barh(range(len(imp)), imp['i'], color='#3498db')
plt.yticks(range(len(imp)), imp['f'])
plt.xlabel('Importancia'); plt.title('Feature Importance - XGBoost')
plt.tight_layout()
plt.savefig(f'{OUT}/09_feature_importance_xgboost.png', dpi=150)
plt.close()
print('Todos os graficos de avaliacao salvos.')


           TABELA COMPARATIVA
       Modelo  Acuracia  Precisao  Recall  F1-Score  AUC-ROC
Random Forest    0.9481    0.8512  0.7935    0.8214   0.9799
      XGBoost    0.9363    0.7309  0.9113    0.8112   0.9818


Todos os graficos de avaliacao salvos.


In [8]:
# ============================================================
# 9. Exportação do modelo e artefatos
# ============================================================
os.makedirs("../model", exist_ok=True)

# Modelo treinado
joblib.dump(xgb_model, "../model/xgboost_falha.joblib")
print("  - ../model/xgboost_falha.joblib")

# Scaler
joblib.dump(scaler, "../model/scaler_falha.joblib")
print("  - ../model/scaler_falha.joblib")

# Feature columns (nomes originais dos sensores)
with open("../model/feature_cols.json", "w") as f:
    json_lib.dump(feature_cols, f)
print("  - ../model/feature_cols.json")

# Feature map com nomes amigáveis (para deploy/interpretação)
feature_map = {
    sensor: {
        "nome_amigavel": nomes_amigaveis[sensor],
        "importancia": round(float(xgb_model.feature_importances_[i]), 4)
    }
    for i, sensor in enumerate(feature_cols)
}
with open("../model/feature_map.json", "w") as f:
    json_lib.dump(feature_map, f, indent=2)
print("  - ../model/feature_map.json")

print("\nTodos os artefatos exportados com sucesso!")


  - ../model/xgboost_falha.joblib
  - ../model/scaler_falha.joblib
  - ../model/feature_cols.json
  - ../model/feature_map.json

Todos os artefatos exportados com sucesso!


# 10. Conclusao\n\nEste projeto aplicou tecnicas de classificacao supervisionada ao dataset CMAPSS FD001 da NASA, que simula a degradacao de motores turbofan ao longo de ciclos de operacao. A analise permitiu comparar dois algoritmos — XGBoost e Random Forest — na tarefa de identificar falhas iminentes (RUL <= 30 ciclos) utilizando apenas 6 sensores selecionados por correlacao com o target.\n\n## Principais descobertas\n\nO Random Forest apresentou o melhor desempenho geral, com acuracia de aproximadamente 96,4% e F1-Score de 0,880, superando o XGBoost (95,5% e 0,857). Ambos os modelos alcancaram AUC-ROC acima de 0,989, indicando excelente capacidade de discriminacao entre as classes. A versao simplificada com 6 sensores manteve performance competitiva (93,4% de acuracia), demonstrando que e possivel reduzir significativamente a dimensionalidade sem perda substancial de qualidade preditiva.\n\nA analise exploratoria revelou que os sensores mais relevantes (sensor_04, sensor_07, sensor_11, sensor_12, sensor_15, sensor_21) estao relacionados a temperatura e pressao do motor, consistente com a fisica da degradacao de turbinas.\n\n## Limitacoes\n\nO dataset CMAPSS e simulado — resultados podem nao refletir dados reais. A classificacao binaria (RUL <= 30) simplifica o problema. O FD001 contempla apenas um modo de falha.\n\n## Aplicacoes em Oleo e Gas\n\nA metodologia pode ser estendida para compressores, turbinas, bombas centrifugas e sistemas de elevacao artificial no setor de petroleo e gas. Prever falhas com antecedencia permite planejar manutencao sem comprometer a producao.\n\n## Melhorias futuras\n\nTrabalhos futuros: (1) RUL como regressao; (2) series temporais (LSTM); (3) dados reais de campo; (4) multiplos modos de falha; (5) interpretabilidade com SHAP.\n\nOs modelos foram exportados e integrados a API FastAPI com interface web de 6 campos, demonstrando aplicabilidade pratica.\n